# CNN Animal Classifier — EfficientNet-B0

This notebook demonstrates animal classification using a **Convolutional Neural Network (CNN)** — specifically EfficientNet-B0 pretrained on ImageNet-1k.

## Architecture Overview
EfficientNet-B0 uses a compound scaling method that uniformly scales network width, depth, and resolution. Key building blocks:
- **MBConv blocks** (Mobile Inverted Bottleneck) with depthwise separable convolutions
- **Squeeze-and-Excitation** modules for channel attention
- **Stochastic depth** for regularization

Pretrained on **ImageNet-1k** (1,000 classes, ~1.28M training images).

In [ ]:
# Install dependencies if needed
# !pip install torch torchvision Pillow numpy matplotlib requests

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.models import EfficientNet_B0_Weights
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import urllib.request
import io
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Pretrained EfficientNet-B0

In [ ]:
# Load EfficientNet-B0 with ImageNet-1k weights
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
model = models.efficientnet_b0(weights=weights)
model = model.to(device)
model.eval()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model size (approx):  {total_params * 4 / 1e6:.1f} MB (float32)')

## 2. Model Architecture Visualization

In [ ]:
# Print top-level architecture
print('EfficientNet-B0 Top-Level Architecture:')
print('=' * 50)
for name, module in model.named_children():
    num_params = sum(p.numel() for p in module.parameters())
    print(f'{name:20s} | params: {num_params:>8,}')

print('\nMBConv Block Stages in "features":')    
print('-' * 50)
for i, layer in enumerate(model.features):
    num_params = sum(p.numel() for p in layer.parameters())
    print(f'features[{i}] {type(layer).__name__:30s} | params: {num_params:>8,}')

## 3. Preprocessing Pipeline

In [ ]:
# Standard ImageNet preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    )
])

def preprocess_image(img: Image.Image) -> torch.Tensor:
    """Preprocess a PIL image for EfficientNet inference."""
    if img.mode != 'RGB':
        img = img.convert('RGB')
    tensor = preprocess(img).unsqueeze(0)  # Add batch dim
    return tensor.to(device)

print('Preprocessing pipeline:')
for t in preprocess.transforms:
    print(f'  {t}')

## 4. Load ImageNet Labels & Define Animal Classes

In [ ]:
# Load ImageNet-1k class labels
IMAGENET_LABELS_URL = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'

try:
    with urllib.request.urlopen(IMAGENET_LABELS_URL) as response:
        imagenet_labels = json.loads(response.read().decode())
    print(f'Loaded {len(imagenet_labels)} ImageNet labels')
except Exception:
    # Fallback: use torchvision weights meta
    imagenet_labels = weights.meta['categories']
    print(f'Loaded {len(imagenet_labels)} labels from torchvision weights')

# Define animal-related ImageNet class indices
# ImageNet has ~400+ animal classes (fish, birds, reptiles, mammals, insects, etc.)
ANIMAL_CLASS_RANGES = [
    (0, 397),    # Fish, birds, reptiles, amphibians, mammals
    (407, 411),  # Some insects (bee, ladybug)
    (665, 668),  # Mosquito, fly
]

ANIMAL_INDICES = set()
for start, end in ANIMAL_CLASS_RANGES:
    ANIMAL_INDICES.update(range(start, end + 1))

print(f'\nAnimal class indices: {len(ANIMAL_INDICES)} classes')
print(f'Sample animal classes: {[imagenet_labels[i] for i in list(ANIMAL_INDICES)[:10]]}')

## 5. Inference Function

In [ ]:
def predict_cnn(image: Image.Image, top_k: int = 5) -> dict:
    """
    Run CNN inference on a PIL image.
    
    Returns:
        dict with keys:
          - 'top_predictions': list of (label, probability) tuples
          - 'all_probabilities': numpy array of 1000 class probs
          - 'predicted_class': top-1 predicted label
          - 'confidence': top-1 confidence score
          - 'is_animal': whether top prediction is an animal class
    """
    tensor = preprocess_image(image)
    
    with torch.no_grad():
        logits = model(tensor)                         # (1, 1000)
        probs = torch.softmax(logits, dim=-1)          # (1, 1000)
        probs_np = probs.squeeze().cpu().numpy()       # (1000,)
    
    top_indices = probs_np.argsort()[::-1][:top_k]
    top_predictions = [(imagenet_labels[i], float(probs_np[i])) for i in top_indices]
    
    top1_idx = top_indices[0]
    return {
        'top_predictions': top_predictions,
        'all_probabilities': probs_np,
        'predicted_class': imagenet_labels[top1_idx],
        'confidence': float(probs_np[top1_idx]),
        'is_animal': top1_idx in ANIMAL_INDICES
    }

print('predict_cnn() function defined.')

## 6. Test on a Sample Animal Image

In [ ]:
# Download a sample tiger image from Wikipedia Commons
SAMPLE_IMAGE_URL = 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Walking_tiger_female.jpg/320px-Walking_tiger_female.jpg'

try:
    with urllib.request.urlopen(SAMPLE_IMAGE_URL) as response:
        img_data = response.read()
    sample_image = Image.open(io.BytesIO(img_data)).convert('RGB')
    print(f'Sample image loaded: {sample_image.size} pixels')
except Exception as e:
    print(f'Could not download sample image: {e}')
    print('Please provide your own image as: sample_image = Image.open("path/to/animal.jpg")')
    sample_image = None

In [ ]:
if sample_image is not None:
    result = predict_cnn(sample_image, top_k=10)
    
    print(f'Predicted class: {result["predicted_class"]}')
    print(f'Confidence:      {result["confidence"]*100:.2f}%')
    print(f'Is animal class: {result["is_animal"]}')
    print()
    print('Top-10 Predictions:')
    print('-' * 40)
    for rank, (label, prob) in enumerate(result['top_predictions'], 1):
        bar = '█' * int(prob * 50)
        print(f'{rank:2d}. {label:30s} {prob*100:6.2f}% {bar}')

## 7. Visualize Predictions

In [ ]:
if sample_image is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: original image
    axes[0].imshow(sample_image)
    axes[0].set_title('Input Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Right: top-10 predictions bar chart
    labels_list = [p[0] for p in result['top_predictions']]
    probs_list  = [p[1] * 100 for p in result['top_predictions']]
    colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(len(labels_list))]

    bars = axes[1].barh(labels_list[::-1], probs_list[::-1], color=colors[::-1])
    axes[1].set_xlabel('Confidence (%)', fontsize=12)
    axes[1].set_title('CNN (EfficientNet-B0) — Top-10 Predictions', fontsize=13, fontweight='bold')
    axes[1].set_xlim(0, max(probs_list) * 1.2)

    for bar, prob in zip(bars[::-1], probs_list):
        axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                     f'{prob:.2f}%', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('../models/cnn_sample_output.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved to models/cnn_sample_output.png')

## 8. Feature Map Visualization (CNN Interpretability)

In [ ]:
if sample_image is not None:
    # Extract feature maps from the first conv layer
    activation = {}

    def get_activation(name):
        def hook(module, input, output):
            activation[name] = output.detach()
        return hook

    # Register hook on the first convolution
    hook_handle = model.features[0][0].register_forward_hook(get_activation('conv1'))

    tensor = preprocess_image(sample_image)
    with torch.no_grad():
        _ = model(tensor)

    hook_handle.remove()

    feature_maps = activation['conv1'].squeeze().cpu().numpy()  # (32, 112, 112)
    print(f'Feature maps shape: {feature_maps.shape}')

    # Plot first 16 feature maps
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    fig.suptitle('CNN First-Layer Feature Maps (EfficientNet-B0)', fontsize=14, fontweight='bold')

    for i, ax in enumerate(axes.flat):
        if i < feature_maps.shape[0]:
            ax.imshow(feature_maps[i], cmap='viridis')
            ax.set_title(f'Filter {i+1}', fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('../models/cnn_feature_maps.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Feature maps saved to models/cnn_feature_maps.png')

## 9. GradCAM — Visualize What the CNN Focuses On

In [ ]:
import torch.nn.functional as F

def compute_gradcam(model, image: Image.Image, target_layer):
    """Compute GradCAM saliency map for the top predicted class."""
    model.eval()
    tensor = preprocess_image(image)  # (1, 3, 224, 224)
    tensor.requires_grad_(True)

    gradients = []
    activations = []

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    def forward_hook(module, input, output):
        activations.append(output)

    fwd_handle = target_layer.register_forward_hook(forward_hook)
    bwd_handle = target_layer.register_backward_hook(backward_hook)

    logits = model(tensor)
    pred_class = logits.argmax(dim=1).item()

    model.zero_grad()
    logits[0, pred_class].backward()

    fwd_handle.remove()
    bwd_handle.remove()

    grads = gradients[0].squeeze().cpu().detach().numpy()    # (C, H, W)
    acts  = activations[0].squeeze().cpu().detach().numpy()  # (C, H, W)

    weights = grads.mean(axis=(1, 2))           # (C,)
    cam = (weights[:, None, None] * acts).sum(axis=0)  # (H, W)
    cam = np.maximum(cam, 0)                    # ReLU
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # normalize

    cam_resized = F.interpolate(
        torch.tensor(cam).unsqueeze(0).unsqueeze(0),
        size=(224, 224), mode='bilinear', align_corners=False
    ).squeeze().numpy()

    return cam_resized, pred_class


if sample_image is not None:
    # Target the last MBConv block's output
    target_layer = model.features[-1]
    cam, pred_idx = compute_gradcam(model, sample_image, target_layer)

    img_resized = sample_image.resize((224, 224))
    img_array   = np.array(img_resized)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_array)
    axes[0].set_title('Original Image', fontsize=13, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(cam, cmap='jet')
    axes[1].set_title('GradCAM Heatmap', fontsize=13, fontweight='bold')
    axes[1].axis('off')

    overlay = img_array.copy().astype(float) / 255.0
    heatmap = plt.cm.jet(cam)[:, :, :3]
    blended = 0.6 * overlay + 0.4 * heatmap
    blended = np.clip(blended, 0, 1)

    axes[2].imshow(blended)
    axes[2].set_title(f'Overlay — CNN predicts: {imagenet_labels[pred_idx]}', fontsize=11, fontweight='bold')
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig('../models/cnn_gradcam.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('GradCAM saved to models/cnn_gradcam.png')

## 10. Save Model Export Utility

This cell saves the preprocessing config and model weights path for use by the Streamlit app.

In [ ]:
import json

cnn_config = {
    'model_name': 'efficientnet_b0',
    'weights': 'EfficientNet_B0_Weights.IMAGENET1K_V1',
    'input_size': 224,
    'num_classes': 1000,
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225],
    'total_params': total_params,
}

with open('../models/cnn_config.json', 'w') as f:
    json.dump(cnn_config, f, indent=2)

print('CNN config saved to models/cnn_config.json')
print(json.dumps(cnn_config, indent=2))

## Summary

| Property | Value |
|---|---|
| Model | EfficientNet-B0 |
| Parameters | ~5.3M |
| Input size | 224 × 224 |
| Output classes | 1,000 (ImageNet-1k) |
| Training data | ImageNet-1k (~1.28M images) |
| Top-1 accuracy | ~77.7% on ImageNet val set |

**Key CNN inductive biases:**
- **Translation equivariance** — detects features regardless of location
- **Local connectivity** — each neuron responds to a local receptive field
- **Hierarchical feature learning** — edges → textures → parts → objects